#### Relevant imports

In [1]:
import csv
import io
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, util
from groq import Groq
from sqlalchemy import create_engine, text, Result
import pandas as pd


**Embedder**  
Embedder model is used from sentence transformer  
This is used for making semantic search

In [2]:
# Load embedder model
# model = SentenceTransformer('all-mpnet-base-v2')
model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Dev\AI\OutSkill\repos\GenAIEngineering-Cohort3-My\week6\venv312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SID\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Similarity Search**  
This function makes the semantic similarity search of a string  
It searches the strings that are closely related to search string by meanining  
The reference strings re-ordered based on the similarity  
If there is a distance threshold provided, only the ones relevant are provided

In [6]:
def semantic_similarity_rank(
    search_string: str,
    sentences: list[str],
    threshold: float = 0.0
) -> tuple[list[str], list[int]]:
    """
    Ranks sentences based on semantic similarity to the search_string.

    Args:
        search_string (str): The input query.
        sentences (list[str]): List of sentences to compare.
        threshold (float): Similarity threshold (0 means no threshold).

    Returns:
        tuple: (reordered_sentences, original_indexes)
    """

    # Encode search string and sentence list
    search_embedding = model.encode(search_string, convert_to_tensor=True)
    sentence_embeddings = model.encode(sentences, convert_to_tensor=True)

    # Compute cosine similarity score
    cosine_scores = util.cos_sim(search_embedding, sentence_embeddings)[0]

    # Pair sentences with scores and original indices
    indexed_scores = [
        (i, s, float(score)) for i, (s, score) in enumerate(zip(sentences, cosine_scores))
        if threshold == 0.0 or float(score) >= threshold
    ]

    # Sort by score descending
    indexed_scores.sort(key=lambda x: x[2], reverse=True)

    # Extract reordered sentences and original indices
    reordered_sentences = [s for _, s, _ in indexed_scores]
    original_indexes = [i for i, _, _ in indexed_scores]

    # Output the reordered senteces and the re-ordered indices in original set
    return reordered_sentences, original_indexes
    
    # # Additional output
    # scores = [sc[2] for sc in indexed_scores]
    # return reordered_sentences, original_indexes, scores


**Read Data File**  
Data File with call record summary read into data frame and the summary column segregated as a list of string  

In [4]:
# Read the CSV and consider the relevant column in a list
Data = pd.read_csv ('call_records.csv')
Summary = Data['Call Summary'].astype (str).tolist ()
# Summary

**Semantic Search**  
For different criteria in the call summary, semantic search is made to capture the indices  
Diff threshold values depending on the criteria

In [5]:
CB_Criteria = "follow up request"
Shortlist, CB_Order = semantic_similarity_rank (CB_Criteria, Summary, 0.4)
print (CB_Order)

CL_Criteria = "clarification sought"
Shortlist, CL_Order = semantic_similarity_rank (CL_Criteria, Summary, 0.3)
print (CL_Order)



[167, 130, 85, 1, 190, 35, 149, 42, 134, 139, 46, 188, 189, 131, 113, 142, 21, 20, 22, 63, 115, 150, 33, 108, 55, 86, 175, 127, 70, 136, 185, 197, 36, 71, 25, 45, 65, 148, 177, 48, 57, 107, 10, 163]
[48, 57, 60, 99, 140, 34, 174, 96, 20, 149, 44, 132, 171, 198, 128, 101, 51, 100, 192, 134, 139, 22, 176, 196, 37, 117, 154, 158, 188, 189, 69, 183, 11, 136, 130, 167, 1, 43, 107, 33, 106, 142, 90, 150, 25, 45, 65, 148, 177, 78, 125, 35, 182, 185, 147, 135, 115, 73, 85, 70, 14, 31, 190]


**Update in CSV**  
Respective records are marked for the criteria match  
This can further be used in the process as structured data file


In [7]:
Data.loc [CB_Order, [CB_Criteria]] = 'Yes'
Data.loc [CL_Order, [CL_Criteria]] = 'Yes'
Data.to_csv ('call_record_updated.csv', index=False)
Data

,Call ID,Call Summary,follow up request,clarification sought
0,C_001,Customer was satisfied with the assistance. Cu...,NaN,NaN
1,C_002,Customer asked about upcoming plan changes. Cu...,Yes,Yes
2,C_003,Customer was unhappy about the billing error.,NaN,NaN
3,C_004,Customer requested help setting up voicemail. ...,NaN,NaN
4,C_005,Customer expressed concern about data speed.,NaN,NaN
...,...,...,...,...
195,C_196,Customer wanted to understand the loyalty bene...,NaN,NaN
196,C_197,Customer questioned the recent charges. Custom...,NaN,Yes
197,C_198,Customer requested help setting up voicemail. ...,Yes,NaN
198,C_199,Customer needed clarification on roaming charges.,NaN,Yes
